In [69]:
import pandas as pd

In [70]:
# Inspeciona o CSV consolidado atual
df = pd.read_csv("../data/processed/inmet_todas_cidades_diario.csv", parse_dates=["data"])

print(f"Shape: {df.shape}")
print(f"\nColunas: {df.columns.tolist()}")
print(f"\nMissing por coluna (%):")
print((df.isnull().mean() * 100).round(1).sort_values(ascending=False).to_string())
print(f"\nAmostra:")
df.head(3)

Shape: (10232, 17)

Colunas: ['data', 'cidade', 'codigo_wmo', 'temp_ar_C_mean', 'temp_max_C_max', 'temp_min_C_min', 'temp_orvalho_C_mean', 'precip_mm_sum', 'radiacao_kJm2_sum', 'umidade_pct_mean', 'umidade_max_pct_max', 'umidade_min_pct_min', 'pressao_mB_mean', 'vento_vel_ms_mean', 'vento_vel_ms_max', 'vento_rajada_ms_max', 'geada']

Missing por coluna (%):
vento_rajada_ms_max    4.0
umidade_min_pct_min    3.9
temp_max_C_max         3.9
umidade_max_pct_max    3.9
temp_min_C_min         3.9
vento_vel_ms_max       3.7
vento_vel_ms_mean      3.7
temp_ar_C_mean         3.6
temp_orvalho_C_mean    3.6
pressao_mB_mean        3.6
umidade_pct_mean       3.6
data                   0.0
cidade                 0.0
radiacao_kJm2_sum      0.0
precip_mm_sum          0.0
codigo_wmo             0.0
geada                  0.0

Amostra:


,data,cidade,codigo_wmo,temp_ar_C_mean,temp_max_C_max,temp_min_C_min,temp_orvalho_C_mean,precip_mm_sum,radiacao_kJm2_sum,umidade_pct_mean,umidade_max_pct_max,umidade_min_pct_min,pressao_mB_mean,vento_vel_ms_mean,vento_vel_ms_max,vento_rajada_ms_max,geada
0,2018-12-31,Machado,A567,21.533333,23.7,20.9,18.000000,0.0,0.0,80.333333,84.0,69.0,905.166667,3.400000,4.1,7.6,0
1,2019-01-01,Machado,A567,22.250000,28.8,18.8,19.008333,6.4,19101.1,82.875000,96.0,57.0,906.675000,1.695833,5.8,11.2,0
2,2019-01-02,Machado,A567,22.966667,28.8,19.3,19.562500,0.0,19186.6,82.208333,97.0,55.0,907.508333,0.750000,4.8,10.0,0


In [71]:
# Colunas a remover e justificativa
REMOVER = [
    "codigo_wmo",            # redundante com cidade
    "temp_orvalho_C_mean",   # colinear com umidade_pct_mean
    "umidade_max_pct_max",   # redundante com umidade_pct_mean
    "umidade_min_pct_min",   # redundante com umidade_pct_mean
    "vento_vel_ms_max",      # colinear com vento_vel_ms_mean e vento_rajada_ms_max
    "pressao_mB_mean",
    'vento_vel_ms_mean',
    'vento_rajada_ms_max',
    'radiacao_kJm2_sum',# efeito capturado por temp + precip
]

df = df.drop(columns=REMOVER)
print(f"Shape após remoção: {df.shape}")
print(f"Colunas restantes: {df.columns.tolist()}")

df

Shape após remoção: (10232, 8)
Colunas restantes: ['data', 'cidade', 'temp_ar_C_mean', 'temp_max_C_max', 'temp_min_C_min', 'precip_mm_sum', 'umidade_pct_mean', 'geada']


,data,cidade,temp_ar_C_mean,temp_max_C_max,temp_min_C_min,precip_mm_sum,umidade_pct_mean,geada
0,2018-12-31,Machado,21.533333,23.7,20.9,0.0,80.333333,0
1,2019-01-01,Machado,22.250000,28.8,18.8,6.4,82.875000,0
2,2019-01-02,Machado,22.966667,28.8,19.3,0.0,82.208333,0
3,2019-01-03,Machado,22.216667,29.4,19.4,4.4,86.583333,0
4,2019-01-04,Machado,20.700000,26.4,18.9,60.4,92.875000,0
...,...,...,...,...,...,...,...,...
10227,2025-12-27,Varginha,NaN,NaN,NaN,0.0,NaN,0
10228,2025-12-28,Varginha,NaN,NaN,NaN,0.0,NaN,0
10229,2025-12-29,Varginha,NaN,NaN,NaN,0.0,NaN,0
10230,2025-12-30,Varginha,NaN,NaN,NaN,0.0,NaN,0


In [72]:
# Interpolação por cidade — mantém coluna cidade
cols_num = df.select_dtypes("number").columns.tolist()

frames = []
for cidade, grupo in df.groupby("cidade"):
    grupo = grupo.sort_values("data").copy()
    for col in cols_num:
        grupo[col] = grupo[col].interpolate(method="linear", limit_direction="both")
    frames.append(grupo)

df = pd.concat(frames).sort_values(["cidade", "data"]).reset_index(drop=True)

print("Missing após interpolação:")
print("  Nenhum missing restante ✓" if not df.isnull().any().any()
      else (df.isnull().mean() * 100).round(2)[df.isnull().any()].to_string())
print(f"\nColunas: {df.columns.tolist()}")
print(f"Shape: {df.shape}")

Missing após interpolação:
  Nenhum missing restante ✓

Colunas: ['data', 'cidade', 'temp_ar_C_mean', 'temp_max_C_max', 'temp_min_C_min', 'precip_mm_sum', 'umidade_pct_mean', 'geada']
Shape: (10232, 8)


In [73]:
# Pivot long → wide (uma linha por dia, colunas por cidade)
cols_vars = [c for c in df.columns if c not in ("data", "cidade")]

df_wide = df.pivot_table(
    index="data",
    columns="cidade",
    values=cols_vars,
    aggfunc="first",
)

# (temp_ar_C_mean, Machado) → temp_ar_C_mean_Machado
df_wide.columns = [f"{var}_{cidade}" for var, cidade in df_wide.columns]
df_wide = df_wide.reset_index().sort_values("data").reset_index(drop=True)

# Remove o dia 2018-12-31 que veio do arquivo de Machado
df_wide = df_wide[df_wide["data"] >= "2019-01-01"].reset_index(drop=True)

print(f"Shape: {df_wide.shape}")
print(f"Período: {df_wide['data'].min().date()} → {df_wide['data'].max().date()}")
print(f"Colunas ({len(df_wide.columns)}): {df_wide.columns.tolist()}")

df_wide.to_csv("../data/processed/inmet_wide_diario.csv", index=False)
print("\nSalvo → data/processed/inmet_wide_diario.csv ✓")

Shape: (2557, 25)
Período: 2019-01-01 → 2025-12-31
Colunas (25): ['data', 'geada_Machado', 'geada_Manhuacu', 'geada_Patrocinio', 'geada_Varginha', 'precip_mm_sum_Machado', 'precip_mm_sum_Manhuacu', 'precip_mm_sum_Patrocinio', 'precip_mm_sum_Varginha', 'temp_ar_C_mean_Machado', 'temp_ar_C_mean_Manhuacu', 'temp_ar_C_mean_Patrocinio', 'temp_ar_C_mean_Varginha', 'temp_max_C_max_Machado', 'temp_max_C_max_Manhuacu', 'temp_max_C_max_Patrocinio', 'temp_max_C_max_Varginha', 'temp_min_C_min_Machado', 'temp_min_C_min_Manhuacu', 'temp_min_C_min_Patrocinio', 'temp_min_C_min_Varginha', 'umidade_pct_mean_Machado', 'umidade_pct_mean_Manhuacu', 'umidade_pct_mean_Patrocinio', 'umidade_pct_mean_Varginha']

Salvo → data/processed/inmet_wide_diario.csv ✓
